# TP final IA II - Paso 6: los modelos en ONNX para la página

La página publicada en Hugging Face (https://huggingface.co/spaces/Final-IA/Final_IA_II) es un Space estático, gratis y sin servidor. Para que el modelo corra igual en vivo, los 12 modelos que exportó el notebook 05 (6 campañas dejadas afuera × hasta enero / campaña completa) se convierten a **ONNX** y la página los ejecuta en el navegador con *onnxruntime-web* (`app/pagina/static/modelo.js`), con el mismo preprocesamiento que `inferencia.py`.

Este notebook hace la conversión y **verifica** que cada modelo ONNX da lo mismo que el de Keras sobre los 192 lotes de ejemplo (`datos_app/ejemplos/series.npz`), con la misma tolerancia que los tests de la API: diferencia máxima de probabilidad menor a 0,01 y la misma clase en al menos el 99,5 % de los lotes. El resultado queda en `datos_app/onnx/`, que es lo que está publicado en la carpeta `onnx/` del Space.

Corre en CPU (no hace falta GPU) y tarda unos minutos.

In [1]:
!pip install -q tf2onnx onnx onnxruntime

In [2]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    APP = '/content/drive/MyDrive/TP_final_IA2/datos_app'
except ImportError:
    APP = os.environ.get('DATOS_APP', 'datos_app')
print(APP, sorted(os.listdir(APP)))

Mounted at /content/drive
/content/drive/MyDrive/TP_final_IA2/datos_app ['curvas_ndvi.npz', 'curvas_tipicas.csv', 'ejemplos', 'inferencia.py', 'lotes.csv', 'metricas.json', 'modelos', 'versiones.json']


In [3]:
import sys, json
import numpy as np
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf, keras, tf2onnx, onnx, onnxruntime as ort
sys.path.insert(0, APP)
from inferencia import preprocesar, CLASES, BANDAS_S2
print('TensorFlow', tf.__version__, '| Keras', keras.__version__, '| tf2onnx', tf2onnx.__version__,
      '| onnx', onnx.__version__, '| onnxruntime', ort.__version__)

SALIDA = os.path.join(APP, 'onnx')
os.makedirs(SALIDA, exist_ok=True)
indice = json.load(open(os.path.join(APP, 'modelos', 'indice.json')))

def convertir(ruta_keras, T, ruta_onnx):
    # La entrada es la serie ya normalizada (n, T, 16, 16, 12); las salidas, la probabilidad de cada clase y el rinde
    m = keras.models.load_model(ruta_keras, compile=False)
    spec = (tf.TensorSpec((None, T, 16, 16, 12), tf.float32, name='serie'),)
    @tf.function(input_signature=spec)
    def f(serie):
        out = m({'serie': serie} if isinstance(m.input, dict) else serie, training=False)
        if not isinstance(out, dict):
            out = dict(zip(m.output_names, out))
        return {'cultivo': tf.cast(out['cultivo'], tf.float32), 'rinde': tf.cast(out['rinde'], tf.float32)}
    tf2onnx.convert.from_function(f, input_signature=spec, opset=17, output_path=ruta_onnx)
    return m

z = np.load(os.path.join(APP, 'ejemplos', 'series.npz'), allow_pickle=True)
x12 = preprocesar(z['bandas'])
print(f'{len(x12)} lotes para verificar')

TensorFlow 2.21.0 | Keras 3.13.2 | tf2onnx 1.17.0 | onnx 1.23.2 | onnxruntime 1.32.0
192 lotes para verificar


In [4]:
filas, nuevo = [], {}
for camp, modos in sorted(indice.items()):
    nuevo[camp] = {}
    for modo, info in modos.items():
        T = info['periodos']
        archivo = info['archivo'].replace('.keras', '.onnx')
        m = convertir(os.path.join(APP, 'modelos', info['archivo']), T, os.path.join(SALIDA, archivo))
        mu = np.asarray(info['mu_canales'], np.float32)
        sd = np.asarray(info['sd_canales'], np.float32)
        x = ((x12[:, :T] - mu) / sd).astype(np.float32)
        k = m.predict({'serie': x} if isinstance(m.input, dict) else x, verbose=0)
        p, r = ort.InferenceSession(os.path.join(SALIDA, archivo), providers=['CPUExecutionProvider']).run(
            ['cultivo', 'rinde'], {'serie': x})
        filas.append({'campaña dejada afuera': camp, 'modelo': modo,
                      'dif. máx. probabilidad': float(np.abs(p - k['cultivo']).max()),
                      'misma clase (%)': 100 * float((p.argmax(1) == k['cultivo'].argmax(1)).mean()),
                      'dif. máx. rinde (kg/ha)': float(np.abs(r.ravel() - k['rinde'].ravel()).max() * info['rinde_desvio']),
                      'MB': os.path.getsize(os.path.join(SALIDA, archivo)) / 1e6})
        nuevo[camp][modo] = {c: v for c, v in info.items() if c != 'archivo'} | {'archivo': archivo}

import pandas as pd
tabla = pd.DataFrame(filas)
pd.set_option('display.float_format', lambda v: f'{v:.2e}' if abs(v) < 1e-2 else f'{v:.2f}')
display(tabla)
ok = (tabla['dif. máx. probabilidad'] < 0.01).all() and (tabla['misma clase (%)'] >= 99.5).all()
print(f"Peor diferencia de probabilidad en los 12 modelos: {tabla['dif. máx. probabilidad'].max():.2e} | "
      f"misma clase: {tabla['misma clase (%)'].min():.1f} % o más | peor diferencia de rinde: "
      f"{tabla['dif. máx. rinde (kg/ha)'].max():.3f} kg/ha")
assert ok, 'algún modelo ONNX no reproduce al de Keras'

,campaña dejada afuera,modelo,dif. máx. probabilidad,misma clase (%),dif. máx. rinde (kg/ha),MB
0,2019_20,completa,2.68e-07,100.00,1.05e-03,0.57
1,2019_20,enero,1.79e-07,100.00,1.05e-03,0.46
2,2020_21,completa,2.98e-07,100.00,9.22e-04,0.57
3,2020_21,enero,3.28e-07,100.00,7.90e-04,0.46
4,2021_22,completa,2.68e-07,100.00,8.16e-04,0.58
5,2021_22,enero,1.79e-07,100.00,1.09e-03,0.46
6,2022_23,completa,3.87e-07,100.00,2.03e-03,0.58
7,2022_23,enero,1.79e-07,100.00,1.02e-03,0.47
8,2023_24,completa,3.58e-07,100.00,1.61e-03,0.58
9,2023_24,enero,2.98e-07,100.00,1.07e-03,0.46


Peor diferencia de probabilidad en los 12 modelos: 3.87e-07 | misma clase: 100.0 % o más | peor diferencia de rinde: 0.002 kg/ha


In [5]:
json.dump({'clases': CLASES, 'bandas': BANDAS_S2, 'modelos': nuevo},
          open(os.path.join(SALIDA, 'indice.json'), 'w'), ensure_ascii=False)
print('Listo:', sorted(os.listdir(SALIDA)))

Listo: ['completa_2019_20.onnx', 'completa_2020_21.onnx', 'completa_2021_22.onnx', 'completa_2022_23.onnx', 'completa_2023_24.onnx', 'completa_2024_25.onnx', 'enero_2019_20.onnx', 'enero_2020_21.onnx', 'enero_2021_22.onnx', 'enero_2022_23.onnx', 'enero_2023_24.onnx', 'enero_2024_25.onnx', 'indice.json']
